In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f9fa'
sns.set_palette("husl")


In [3]:
def load_data(filepath: str):
    df = pd.read_csv(filepath, parse_dates=['Timestamp'])
    df = df.sort_values('Timestamp').reset_index(drop=True)
    df.set_index('Timestamp', inplace=True)

    print(f"Dataset shape     : {df.shape}")
    print(f"Date range        : {df.index.min()} → {df.index.max()}")
    print(f"Total days        : {(df.index.max() - df.index.min()).days}")
    print(f"Missing values:\n{df.isnull().sum()}")

    print(f"\nBasic stats (Load Demand kW):")
    print(df['Load Demand (kW)'].describe().round(2))

    return df


df = load_data('/content/drive/MyDrive/Colab Notebooks/electricity_demand_srilanka.csv')

Dataset shape     : (189888, 15)
Date range        : 2020-01-01 00:00:00 → 2025-05-31 23:45:00
Total days        : 1977
Missing values:
Temperature (°C)               0
Humidity (%)                   0
Wind Speed (m/s)               0
Rainfall (mm)                  0
Solar Irradiance (W/m²)        0
GDP (LKR)                      0
Per Capita Energy Use (kWh)    0
Electricity Price (LKR/kWh)    0
Day of Week                    0
Hour of Day                    0
Month                          0
Season                         0
Public Event                   0
Load Demand (kW)               0
Poya Day                       0
dtype: int64

Basic stats (Load Demand kW):
count    189888.00
mean       1500.15
std         199.93
min         606.88
25%        1365.22
50%        1500.27
75%        1635.07
max        2412.42
Name: Load Demand (kW), dtype: float64


In [18]:
import os
def plot_overview(df: pd.DataFrame):
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.suptitle('Load Demand Overview — Sri Lanka', fontsize=16, fontweight='bold')

    # Full time series
    df['Load Demand (kW)'].plot(ax=axes[0], color='#2196F3', linewidth=0.5, alpha=0.8)
    axes[0].set_title('Full Time Series (15-min intervals)')
    axes[0].set_ylabel('Load Demand (kW)')

    # One week sample
    sample = df.iloc[:672]  # 96 intervals/day × 7 days
    sample['Load Demand (kW)'].plot(ax=axes[1], color='#4CAF50', linewidth=1.2)
    axes[1].set_title('Sample Week — Daily Pattern Visible')
    axes[1].set_ylabel('Load Demand (kW)')

    # Distribution
    axes[2].hist(df['Load Demand (kW)'], bins=80, color='#FF9800', edgecolor='white', linewidth=0.3)
    axes[2].set_title('Distribution of Load Demand')
    axes[2].set_xlabel('Load Demand (kW)')
    axes[2].set_ylabel('Frequency')

    plt.tight_layout()
    os.makedirs('outputs', exist_ok=True) # Create the outputs directory if it doesn't exist
    plt.savefig('outputs/01_overview.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/01_overview.png")


In [19]:
def plot_demand_heatmap(df: pd.DataFrame):
    pivot = df.pivot_table(
        values='Load Demand (kW)',
        index='Hour of Day',
        columns='Day of Week',
        aggfunc='mean'
    )
    day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    pivot.columns = day_labels[:len(pivot.columns)]

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(
        pivot, ax=ax, cmap='YlOrRd', annot=False,
        fmt='.0f', linewidths=0.3,
        cbar_kws={'label': 'Avg Load Demand (kW)'}
    )
    ax.set_title('Average Load Demand: Hour of Day × Day of Week\n(Key pattern for LSTM/CNN-LSTM)', fontsize=14)
    ax.set_xlabel('Day of Week')
    ax.set_ylabel('Hour of Day (0 = Midnight)')
    plt.tight_layout()
    plt.savefig('outputs/02_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/02_heatmap.png")
    print("\nPeak demand hour:", pivot.values.argmax() // len(pivot.columns))


In [21]:
def plot_seasonal(df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # By season
    season_order = ['Summer', 'Winter', 'Fall', 'Spring']
    season_data = df.groupby('Season')['Load Demand (kW)'].mean().reindex(
        [s for s in season_order if s in df['Season'].unique()]
    )
    season_data.plot(kind='bar', ax=axes[0], color=['#FF5722', '#2196F3', '#FF9800', '#4CAF50'],
                     edgecolor='white', linewidth=0.5)
    axes[0].set_title('Average Load Demand by Season')
    axes[0].set_xlabel('Season')
    axes[0].set_ylabel('Avg Load Demand (kW)')
    axes[0].tick_params(axis='x', rotation=0)

    # By month
    monthly = df.groupby('Month')['Load Demand (kW)'].mean()
    monthly.plot(kind='line', ax=axes[1], marker='o', color='#9C27B0', linewidth=2, markersize=6)
    axes[1].set_title('Average Load Demand by Month')
    axes[1].set_xlabel('Month')
    axes[1].set_ylabel('Avg Load Demand (kW)')
    axes[1].set_xticks(range(1, 13))
    axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                              'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Seasonal Demand Patterns — Sri Lanka', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/03_seasonal.png', dpi=150, bbox_inches='tight')
    plt.show()


In [15]:
def plot_weather_correlation(df: pd.DataFrame):
    weather_cols = [
        'Temperature (°C)', 'Humidity (%)', 'Wind Speed (m/s)',
        'Rainfall (mm)', 'Solar Irradiance (W/m²)', 'Load Demand (kW)'
    ]
    corr = df[weather_cols].corr()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Correlation matrix
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=axes[0], mask=mask, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                linewidths=0.5, cbar_kws={'shrink': 0.8})
    axes[0].set_title('Feature Correlation Matrix')

    # Temperature vs Load scatter
    axes[1].scatter(df['Temperature (°C)'], df['Load Demand (kW)'],
                    alpha=0.1, s=1, color='#FF5722')
    z = np.polyfit(df['Temperature (°C)'], df['Load Demand (kW)'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df['Temperature (°C)'].min(), df['Temperature (°C)'].max(), 100)
    axes[1].plot(x_line, p(x_line), 'b-', linewidth=2, label='Trend')
    corr_val = df['Temperature (°C)'].corr(df['Load Demand (kW)'])
    axes[1].set_title(f'Temperature vs Load Demand\n(Pearson r = {corr_val:.3f})')
    axes[1].set_xlabel('Temperature (°C)')
    axes[1].set_ylabel('Load Demand (kW)')
    axes[1].legend()

    plt.suptitle('Weather Impact on Electricity Demand', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/04_weather_corr.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\nCorrelations with Load Demand:")
    print(corr['Load Demand (kW)'].drop('Load Demand (kW)').sort_values(ascending=False))


In [16]:
def plot_poya_effect(df: pd.DataFrame):
    if 'Poya Day' not in df.columns:
        print("Poya Day column not found — skipping.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Average demand: Poya vs non-Poya
    poya_avg   = df[df['Poya Day'] == 1]['Load Demand (kW)'].mean()
    normal_avg = df[df['Poya Day'] == 0]['Load Demand (kW)'].mean()
    axes[0].bar(['Normal Day', 'Poya Day'], [normal_avg, poya_avg],
                color=['#2196F3', '#FF9800'], edgecolor='white', linewidth=0.5)
    axes[0].set_title(f'Average Load: Poya vs Normal\nDifference = {(normal_avg - poya_avg):.1f} kW')
    axes[0].set_ylabel('Avg Load Demand (kW)')

    # Hour-by-hour comparison
    poya_hourly   = df[df['Poya Day'] == 1].groupby('Hour of Day')['Load Demand (kW)'].mean()
    normal_hourly = df[df['Poya Day'] == 0].groupby('Hour of Day')['Load Demand (kW)'].mean()
    axes[1].plot(normal_hourly.index, normal_hourly.values, 'b-o', markersize=4, label='Normal Day', linewidth=1.5)
    axes[1].plot(poya_hourly.index, poya_hourly.values, 'o-', color='#FF9800', markersize=4,
                 label='Poya Day', linewidth=1.5)
    axes[1].set_title('Hour-by-Hour Demand: Poya vs Normal')
    axes[1].set_xlabel('Hour of Day')
    axes[1].set_ylabel('Avg Load Demand (kW)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Poya Day Effect on Electricity Demand — Sri Lanka Specific', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/05_poya_effect.png', dpi=150, bbox_inches='tight')
    plt.show()


In [17]:
def detect_anomalies(df: pd.DataFrame, threshold: float = 3.0) -> pd.DataFrame:
    df = df.copy()
    df['z_score'] = np.abs(stats.zscore(df['Load Demand (kW)'].fillna(df['Load Demand (kW)'].mean())))
    anomalies = df[df['z_score'] > threshold]
    print(f"\nAnomalies detected (|z| > {threshold}): {len(anomalies)}")
    print(f"Anomaly rate: {len(anomalies)/len(df)*100:.2f}%")

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(df.index, df['Load Demand (kW)'], color='#2196F3', linewidth=0.5, alpha=0.7, label='Normal')
    ax.scatter(anomalies.index, anomalies['Load Demand (kW)'],
               color='red', s=10, zorder=5, label=f'Anomaly (z>{threshold})')
    ax.set_title('Load Demand Anomaly Detection (Z-Score Method)')
    ax.set_ylabel('Load Demand (kW)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('outputs/06_anomalies.png', dpi=150, bbox_inches='tight')
    plt.show()

    return anomalies
